<a href="https://colab.research.google.com/github/anacvetkoo/MOS-projekt/blob/main/notebooks/Ziva_teden6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Živa – 6. teden: statistični test (RV3)

**RV3:** Kako so se spreminjali stroški študija ob upoštevanju inflacije?

**Zakaj ta test?** V prejšnji analizi so se razlike v realni rasti šolnin med skupinami zmanjšale, ko sem primerjala isti nabor univerz. Zato zdaj primerjam rast šolnin posameznih univerz.

**Podvprašanje:** Ali se realna odstotna rast šolnin (2004–2023) statistično značilno razlikuje med Ivy League in drugimi zasebnimi neprofitnimi štiriletnimi univerzami?

**Viri:** IPEDS (šolnine in univerze), CPI-U/BLS prek FRED (inflacija; USD leta 2023).

## 1. Hipotezi in vrste podatkov

- **Neodvisna spremenljivka:** skupina univerze – nominalna (Ivy League / druge zasebne neprofitne).
- **Odvisna spremenljivka:** realna odstotna rast šolnine – numerična.
- **Vzorca:** nepovezani skupini; vsaka univerza prispeva eno vrednost.

**H0:** Med skupinama ni razlike v rangih realne rasti šolnin.  
**H1:** Med skupinama je razlika v rangih realne rasti šolnin.

**Prag pomembnosti:** α = 0,05. Za izbiro testa preverim normalnost porazdelitve.


In [ ]:
# Priprava okolja in uvoz podatkov

import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# Mapa s podatki
mapa = Path("/content/drive/MyDrive/FERI/3. letnik/1. semester/MOS/Projekt")

# Podatki o šolninah
df_ipeds = pd.read_csv(
    mapa / "IPEDS_RV3.csv",
    usecols=["UNITID", "leto", "CHG3AY3"]
)

# Podatki o univerzah
df_institucije = pd.read_csv(
    mapa / "IPEDS_institucije_2004_2024.csv",
    usecols=["UNITID", "INSTNM", "leto", "SECTOR"]
)

# Preverjanje podatkov
print("Zapisi o šolninah:", len(df_ipeds))
print("Zapisi o univerzah:", len(df_institucije))
print("Obdobje:", df_ipeds["leto"].min(), "–", df_ipeds["leto"].max())

Mounted at /content/drive
Zapisi o šolninah: 88205
Zapisi o univerzah: 147769
Obdobje: 2004 – 2023


In [ ]:
# RV3: Izbira univerz za statistični test

# Identifikatorji osmih univerz Ivy League
ivy_unitid = [
    217156, 190150, 190415, 182670,
    166027, 186131, 215062, 130794
]

# Povežem podatke o šolninah in univerzah
df_rv3 = df_ipeds.merge(
    df_institucije[["UNITID", "leto", "SECTOR"]],
    on=["UNITID", "leto"],
    how="left",
    validate="one_to_one"
)

# Izberem leti 2004 in 2023 ter zasebne neprofitne univerze
df_rv3 = df_rv3[
    (df_rv3["leto"].isin([2004, 2023])) &
    (df_rv3["SECTOR"] == 2) &
    (df_rv3["CHG3AY3"] > 0)
].copy()

# Razdelim univerze v dve skupini
df_rv3["Skupina"] = np.where(
    df_rv3["UNITID"].isin(ivy_unitid),
    "Ivy League",
    "Druge zasebne neprofitne"
)

# Obdržim samo univerze s podatki za obe leti
df_rv3 = df_rv3.groupby("UNITID").filter(
    lambda x: x["leto"].nunique() == 2
)

# Preverim število univerz
print("Število univerz po skupinah:")
print(df_rv3.groupby("Skupina")["UNITID"].nunique())

Število univerz po skupinah:
Skupina
Druge zasebne neprofitne    1069
Ivy League                     8
Name: UNITID, dtype: int64


In [ ]:
# RV3: Izračun realne odstotne rasti šolnin

# Podatki o inflaciji (CPI-U)
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCNS"
df_cpi = pd.read_csv(url)

stolpec = "observation_date" if "observation_date" in df_cpi.columns else "DATE"
df_cpi["leto"] = pd.to_datetime(df_cpi[stolpec]).dt.year

# Letno povprečje indeksa CPI
cpi_letni = df_cpi.groupby("leto")["CPIAUCNS"].mean()
cpi_2004 = cpi_letni.loc[2004]
cpi_2023 = cpi_letni.loc[2023]

# Šolnini za obe leti postavim v ločena stolpca
df_test = df_rv3.pivot(
    index=["UNITID", "Skupina"],
    columns="leto",
    values="CHG3AY3"
).reset_index()

df_test = df_test.rename(columns={
    2004: "Solnina_2004",
    2023: "Solnina_2023"
})

# Šolnino 2004 pretvorim v USD leta 2023
df_test["Realna_solnina_2004"] = (
    df_test["Solnina_2004"] * cpi_2023 / cpi_2004
)

# Izračun realne odstotne rasti za vsako univerzo
df_test["Realna_rast_pct"] = (
    (df_test["Solnina_2023"] /
     df_test["Realna_solnina_2004"] - 1) * 100
)

# Preverim rezultat
print("Število univerz:", len(df_test))
print("Manjkajoče vrednosti:",
      df_test["Realna_rast_pct"].isna().sum())

print("\nRealna rast šolnin Ivy League:")
display(
    df_test[df_test["Skupina"] == "Ivy League"].round(1)
)

Število univerz: 1077
Manjkajoče vrednosti: 0

Realna rast šolnin Ivy League:


leto,UNITID,Skupina,Solnina_2004,Solnina_2023,Realna_solnina_2004,Realna_rast_pct
106,130794,Ivy League,29820.0,64700.0,48104.8,34.5
376,166027,Ivy League,30620.0,59076.0,49395.4,19.6
514,182670,Ivy League,30465.0,65739.0,49145.3,33.8
531,186131,Ivy League,29910.0,59710.0,48250.0,23.8
553,190150,Ivy League,31472.0,69045.0,50769.8,36.0
555,190415,Ivy League,30167.0,66014.0,48664.6,35.7
815,215062,Ivy League,30716.0,66104.0,49550.2,33.4
842,217156,Ivy League,30672.0,68230.0,49479.3,37.9


In [ ]:
# RV3: Opisne statistike realne rasti šolnin

opisne = df_test.groupby("Skupina")["Realna_rast_pct"].describe()

# Izberem najpomembnejše statistike
opisne = opisne[["count", "min", "25%", "50%", "75%", "max"]]

# Preimenujem stolpce
opisne = opisne.rename(columns={
    "count": "Število univerz",
    "min": "Minimum",
    "25%": "Q1",
    "50%": "Mediana",
    "75%": "Q3",
    "max": "Maksimum"
})

display(opisne.round(1))

,Število univerz,Minimum,Q1,Mediana,Q3,Maksimum
Skupina,,,,,,
Druge zasebne neprofitne,1069.0,-58.1,18.0,32.0,45.0,5825.8
Ivy League,8.0,19.6,31.0,34.1,35.7,37.9


Pred statističnim testom sem izračunala opisne statistike obeh skupin (mediano, kvartile, minimum in maksimum), da sem preverila značilne vrednosti in razpršenost realne rasti šolnin. Osredotočila sem se na mediano, ker je manj občutljiva na izrazite vrednosti kot povprečje.

**Sklep o podatkih:**
- Imam 8 univerz Ivy League in 1.069 drugih zasebnih neprofitnih univerz (brez manjkajočih vrednosti).
- Mediana realne rasti šolnin je pri Ivy League nekoliko višja (34,1 % proti 32,0 %).
- Realna rast šolnin je pri drugih zasebnih neprofitnih univerzah precej bolj razpršena.

In [ ]:
# RV3: Preverjanje največjih realnih rasti šolnin

# Poiščem 5 univerz z največjo realno rastjo
odstopanja = df_test.nlargest(5, "Realna_rast_pct")

# Dodam imena univerz iz leta 2023
imena = df_institucije[
    df_institucije["leto"] == 2023
][["UNITID", "INSTNM"]]

odstopanja = odstopanja.merge(imena, on="UNITID", how="left")

display(
    odstopanja[[
        "INSTNM", "Skupina", "Solnina_2004",
        "Solnina_2023", "Realna_rast_pct"
    ]].round(1)
)

,INSTNM,Skupina,Solnina_2004,Solnina_2023,Realna_rast_pct
0,Berea College,Druge zasebne neprofitne,516.0,49326.0,5825.8
1,Jewish Theological Seminary of America,Druge zasebne neprofitne,11340.0,65545.0,258.3
2,Baptist Missionary Association Theological Sem...,Druge zasebne neprofitne,2560.0,10050.0,143.4
3,AdventHealth University,Druge zasebne neprofitne,5555.0,20880.0,133.0
4,Bryn Athyn College of the New Church,Druge zasebne neprofitne,7410.0,27804.0,132.6


Preverila sem univerze z največjo realno rastjo šolnin, da bi ugotovila, katere univerze najbolj izstopajo.

**Sklep o podatkih:**
- Najbolj izstopa Berea College (+5.825,8 %), zato je pri interpretaciji potrebna previdnost. Podatek sem ohranila, saj brez dodatnega preverjanja ne morem sklepati, da gre za napako.

## 2. Preverjanje normalnosti in izbira testa

**Shapiro-Wilkov** test:

**H0:** normalna porazdelitev.

**H1:** odstopanje od normalnosti.

Če je p < 0,05, H0 zavrnem.

In [ ]:
# Preverjanje normalnosti – Shapiro-Wilkov test

# Pripravim podatke obeh skupin
x = df_test[df_test["Skupina"] == "Ivy League"]["Realna_rast_pct"]
y = df_test[df_test["Skupina"] == "Druge zasebne neprofitne"]["Realna_rast_pct"]

# Izvedem Shapiro-Wilkov test za vsako skupino
shapiro_ivy = stats.shapiro(x)
shapiro_druge = stats.shapiro(y)

print("Ivy League:")
display(shapiro_ivy)

print("\nDruge zasebne neprofitne univerze:")
display(shapiro_druge)

Ivy League:


ShapiroResult(statistic=np.float64(0.7963415935226485), pvalue=np.float64(0.026153918360221125))


Druge zasebne neprofitne univerze:


ShapiroResult(statistic=np.float64(0.06501533786914526), pvalue=np.float64(7.225647611491593e-58))

**Sklep:**
* Ivy League: p = 0,0262
* druge univerze: p < 0,001.

Ker je pri obeh skupinah p < 0,05, zavrnem H0 o normalnosti.

Izberem neparametrični **Mann-Whitneyjev U-test**, ker primerjam dve nepovezani skupini, podatki pa odstopajo od normalne porazdelitve in vsebujejo izrazite vrednosti. Test temelji na rangih in ne zahteva normalnosti.


In [ ]:
# RV3: Mann-Whitneyjev U-test

# Pripravim podatke
x = df_test[df_test["Skupina"] == "Ivy League"]["Realna_rast_pct"]
y = df_test[df_test["Skupina"] == "Druge zasebne neprofitne"]["Realna_rast_pct"]

# Izvedem Mann-Whitneyjev U-test
mw_test = stats.mannwhitneyu(
    x, y,
    alternative="two-sided",  # Dvostranski test – preverjam razlike v obe smeri
    method="asymptotic"       # Približen izračun p-vrednosti, ki upošteva tudi vezane range
)

# Izpis rezultatov
display(mw_test)
print("U-statistika:", mw_test.statistic)
print("p-vrednost:", mw_test.pvalue)

MannwhitneyuResult(statistic=np.float64(4263.0), pvalue=np.float64(0.9886215583059799))

U-statistika: 4263.0
p-vrednost: 0.9886215583059799


## 3. Rezultat in sklep

**Mann-Whitneyjev U-test:**
- U = 4263,0
- p = 0,9886
- α = 0,05

**U-statistika:** Primerjala sem 8 univerz Ivy League z 1.069 drugimi univerzami, kar pomeni 8.552 možnih primerjav. U = 4263 pomeni, da je imela Ivy League višjo realno rast šolnin v približno 49,8 % primerjav (pri enakih vrednostih se upošteva polovica). Ker je to skoraj 50 %, nobena skupina nima izrazito višje rasti šolnin glede na te primerjave.

Ker je **p > 0,05**, ničelne hipoteze **ne zavrnem**.

**Sklep:** Test ni pokazal statistično značilne razlike v rangih realne rasti šolnin med Ivy League in drugimi zasebnimi neprofitnimi univerzami.

### Omejitve
* Skupini sta zelo različnih velikosti (8 in 1.069) in različne razpršenosti.
* Univerze niso naključni vzorec.
* Rezultat ne dokazuje, da sta porazdelitvi ali mediani enaki.

### Povezava z RV3
Test nadgrajuje dosedanjo opisno analizo šolnin. Primerjava s College Board (Ma in Pender, 2022) predstavlja širši kontekst, saj uporablja drugačno agregiranje in obdobje podatkov.